# Transcription examples
This notebook is a collection of examples using the AMPAV libraries

## Install AMPAV
First, the ampav python packages will need to be installed. The command below will take some time to run -- it will need to download many support packages.  Luckily, this will only need to be done once.  If it is run more than than once it won't make any changes but it will run faster.

In [1]:
%pip install --extra-index-url https://pypi.dlib.indiana.edu ampav-core ampav-whisper ampav-parakeet

Looking in indexes: https://pypi.org/simple, https://pypi.dlib.indiana.edu

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Getting technical metadata for a media file
You can retrieve the technical metadata (such as container, codecs, duration, etc) for a media file by loading it into the `AVMetadata` schema. 

In [2]:
from ampav.core.schema.av_metadata import AVMetadata
avm = AVMetadata.from_file("gettysburg.wav")
print(f"media format: {avm.format_name}")
print(f"media duration (in milliseconds): {avm.duration}")
print(f"number of audio streams: {len(avm.streams.audio)}")
print(f"number of video streams: {len(avm.streams.video)}")

media format: wav
media duration (in milliseconds): 183040000.0
number of audio streams: 1
number of video streams: 0


Besides querying individual values, you can dump the whole structure as YAML which is semi-human readable.  You can learn more about the YAML on [Wikipedia](https://en.wikipedia.org/wiki/YAML).  

In [3]:
print(avm.model_dump_yaml())

ampav_format: avmetadata
ampav_format_version: 1
bit_rate: 256004
duration: 183040000.0
format_long_name: WAV / WAVE (Waveform Audio)
format_name: wav
metadata:
  encoder: Lavf60.3.100
  genre: Blues
size: 5857372
streams:
  audio:
  - bit_rate: 256000
    channels: 1
    codec_name: pcm_s16le
    codec_tag: "\x01\0\0\0"
    duration: 183.04
    index: 0
    layout: 1 channels
    metadata: {}
    sample_rate: 16000
  subtitle: []
  video: []



# Tools
Many AMPAV packages implement functions which generate a `ToolOutput` object.  The `ToolObject` will wrap information about what generated the metadata output as well as the metadata itself.

A simple tool is creating a transcript from plain text.  Let's create a transcript from the `gettysburg.txt` file and look at it

In [4]:
from ampav.core.formats.transcript.textfile import import_text_transcript
from pathlib import Path

txt = Path("gettysburg.txt").read_text()
xscript = import_text_transcript(txt)

The warning above is because words in a text file do not have explicit start and end times so it's warning that any timestamps may not be valid.

Let's look at the tool information:

In [5]:
print(f"Tool name: {xscript.tool_name}, version: {xscript.tool_version}")

Tool name: text import, version: 0.0.0


The tool output, a `Transcript` in this case is in the `output` property.  We can assign it to a variable to make it easier to access:

In [6]:
ts = xscript.output
print(f"Start of transcript: {ts.text[:80]}...")
print(f"The first and last word of the transcript: {ts.words[0].word}, {ts.words[-1].word}")
print(f"Start and stop time of the first word: {ts.words[0].start_time}, {ts.words[0].end_time}")

Start of transcript: 140 years ago today President Abraham Lincoln stood at the site of one of the mo...
The first and last word of the transcript: 140, 2009
Start and stop time of the first word: None, None


Since our content came from a text file, the start and stop words are (unsurprisingly) `None`.

## Transcript using OpenAI Whisper
The package `ampav.whisper` provides the code needed to provide a transcript using Whisper.

This will use your CPU for the transcription so it will be a little slow, and since we're using the `small` model, the output may not be great.  You can set the device to "cuda" if you have a GPU in your computer.

## A note on GPU memory...
NOTE: If you start getting GPU out of memory issues you will need to stop your Kernel to clear the memory.  This is true even if you've been trying to force it to CPU usage.

In [7]:
import ampav.whisper.transcribe as whisperts
xscript2 = whisperts.transcribe_file("gettysburg.wav", "small", device="cpu")

/data/work_projects/AMPAV-JupyterLab/transcription-notebooks/.venv/lib64/python3.12/site-packages/whisper/transcribe.py:130: UserWarning: Performing inference on CPU when CUDA is available
  warnings.warn("Performing inference on CPU when CUDA is available")
/data/work_projects/AMPAV-JupyterLab/transcription-notebooks/.venv/lib64/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


You might get some warnings about FP16 or FP32 -- you can ignore these since they come from the underlying tools complaining about your computer.

Let's look at the same data we did for the text file import:

In [8]:
print(f"Tool name: {xscript2.tool_name}, version: {xscript2.tool_version}")
ts = xscript2.output
print(f"Start of transcript: {ts.text[:80]}...")
print(f"The first and last word of the transcript: {ts.words[0].word}, {ts.words[-1].word}")
print(f"Start and stop time of the first word: {ts.words[0].start_time}, {ts.words[0].end_time}")

Tool name: whisper, version: None
Start of transcript: 140 years ago today, President Abraham Lincoln stood at the site of one of the m...
The first and last word of the transcript: 140, ago
Start and stop time of the first word: 0.0, 1.08


This time there's a real start and stop time for the first word.  But wait...why is the last word 'ago'?  If you look at the gettysburg.txt the last word is really 2009!

Let's look at the text of the transcript

In [9]:
print(ts.text)

140 years ago today, President Abraham Lincoln stood at the site of one of the most important battles of the U.S. Civil War in Gettysburg, Pennsylvania. A crowd of some 15,000 had gathered to consecrate a cemetery for Union war dead. Though brief, the now famous Gettysburg Address marked a defining moment of the Civil War. Actor Sam Waterston has portrayed President Lincoln on stage and screen. Four score and seven years ago.


Well, it looks like whisper's small model cut it off.  Good job, whisper.  

## Transcription using parakeet
We can do the same transcription using parakeet.  We'll need to load the package and then run it.

If you have a GPU you can set `cpu_only` to `False`.  You may also need to set some additional parameters to allow the audio to fit in your GPU memory:  `chunk_size` is the size of audio (in seconds) to process at once, and `chunk_overlap` is how many seconds of overlap the audio chunks have with each other. The defaults are 30 and 5, respectively.

In [1]:
import ampav.parakeet.transcribe as parakeetts
xscript3 = parakeetts.transcribe_file("gettysburg.wav", cpu_only=True)

/data/work_projects/AMPAV-JupyterLab/transcription-notebooks/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[NeMo W 2026-04-17 13:44:03 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


[NeMo I 2026-04-17 13:44:07 mixins:184] Tokenizer SentencePieceTokenizer initialized with 8192 tokens


[NeMo W 2026-04-17 13:44:09 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 10.0
    min_duration: 1.0
    text_field: answer
    batch_duration: null
    max_tps: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-04-17 13:44:09 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation 

[NeMo I 2026-04-17 13:44:13 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-04-17 13:44:13 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2026-04-17 13:44:13 label_looping_base:125] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2026-04-17 13:44:13 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2026-04-17 13:44:13 label_looping_base:125] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2026-04-17 13:44:14 save_restore_connector:285] Model EncDecRNNTBPEModel was successfully restored from /home/bdwheele/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v3/snapshots/6d590f77001d318fb17a0b5bf7ee329a91b52598/parakeet-tdt-0.6b-v3.nemo.
[NeMo I 2026-04-17 13:44:14 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2026-04-17 13:44:14 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2026-04-17 13:44:14 label_looping_base:125] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2026-04-17 13:44:16 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2026-04-17 13:44:19 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2026-04-17 13:44:22 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2026-04-17 13:44:25 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2026-04-17 13:44:27 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis o

The nemo engine that runs parakeet will generate a lot of log messages.  This is normal.  However, if it complains that it has run out of GPU memory (even if you're trying to force it to use CPU), you will need to restart your kernel and restart at this step.

Let's look at the same properties that we've been checking before:

In [4]:
print(f"Tool name: {xscript3.tool_name}, version: {xscript3.tool_version}")
ts = xscript3.output
print(f"Start of transcript: {ts.text[:80]}...")
print(f"The first and last word of the transcript: {ts.words[0].word}, {ts.words[-1].word}")
print(f"Start and stop time of the first word: {ts.words[0].start_time}, {ts.words[0].end_time}")

Tool name: parakeet, version: None
Start of transcript: Four score and seven years ago our fathers brought forth on this continent a new...
The first and last word of the transcript: Four, 2009
Start and stop time of the first word: 25.736000000000022, 26.136000000000024


It looks like parakeet got the last word of our transcript correct, but somehow didn't transcribe the beginning.  

## Doing something with a transcript
Let's use one of the transcripts and make a WebVTT from it.  The code below is using the parakeet transcript, but you can generate the others by changing the variable:  `xscript` is from the text file, `xscript2` is from whisper, `xscript3` is from Parakeet.

The paragraphs_to_webvtt function takes list of paragraphs and makes them into a WebVTT file.  The transcription tools will generate a set of paragraphs (with start/stop times if they're available) when the transcript is generated.

In [ ]:
from ampav.core.formats.transcript.webvtt import paragraphs_to_webvtt
print(paragraphs_to_webvtt(xscript3.output.paragraphs))

# Storing content for future use

All of the schemas in AMPAV are designed to allow writing them to storage and loading them later.  Let's load the text transcript again, save it, and then load it again to see if it's the same.  The `import_text_transcript` creates a `ToolOutput` object.

In [3]:
from pathlib import Path
from ampav.core.formats.transcript.textfile import import_text_transcript
from ampav.core.schema import ToolOutput
import yaml

# generate the transcript ToolOutput
text = Path("gettysburg.txt").read_text()
one = import_text_transcript(text)

# save the ToolOutput
with open("saved-transcript.yaml", "w") as f:
    f.write(one.model_dump_yaml())

#  load the ToolOutput yaml file
# note to self:  probably should make a utility function to make this more straightforward
two = ToolOutput(**yaml.safe_load(Path("saved-transcript.yaml").read_text()))

print(one == two)
    


True


The result should be `True` -- by default Python does a deep comparison.  You can look at the `saved-transcript.yaml` file to see what the saved output looks like.